# Week 10 lab · An unfair die and real CO₂ data

**CMT309/509 · CMT124** · two-hour lab

**Part A** simulates a biased die: you build it from uniform random numbers (the inverse-CDF method), watch simulations converge to the theory, and draw a running 95 % confidence band. **Part B** applies the same ideas to real data: confidence intervals and a Welch t-test for CO₂ emissions per person in two regions.

**How this lab works**
* Work through the tasks in order. Each task has an empty cell for your code, followed by a **checkpoint** cell.
* Run the checkpoint after your code. If it prints `✔`, move on. If it raises an `AssertionError`, read the message, fix your code and try again.
* If you are stuck on a task that later tasks need, run the cell marked *If your result does not match the checkpoint* to continue.
* **Question** cells ask you to explain a result in a sentence or two. Write your answer in the text cell (double-click to edit).
* The **Challenge** at the end is optional.

You need this week's lecture (probability, PMF/CDF, inverse CDF, confidence intervals, t-tests) plus numpy (Week 5), pandas basics and matplotlib.

In [ ]:
# --- Setup: run this cell first -------------------------------------------------
# Course datasets live in the course repository under 2026-27/datasets/.
# get_data() uses a local copy if the notebook runs inside a copy of the repository;
# otherwise (e.g. in Google Colab) it downloads the file once and reuses it.
import os
import urllib.request

COURSE_REPO = "OWNER/REPO"
BASE_URL = f"https://raw.githubusercontent.com/{COURSE_REPO}/main/2026-27/"


def course_file(relpath):
    """Return a local path to a course file such as 'datasets/owid_co2_cleaned.csv'."""
    for root in ("../../..", "."):                     # 1) running inside a copy of the repository
        path = os.path.join(root, relpath)
        if os.path.exists(path):
            return path
    path = os.path.join("course_files", relpath)       # 2) Colab: download once
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        urllib.request.urlretrieve(BASE_URL + relpath, path)
    return path


def get_data(name):
    """Return a local path to a course dataset, e.g. get_data('owid_co2_cleaned.csv')."""
    return course_file("datasets/" + name)


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(2026)

# Part A · An unfair die

This die favours the high faces:

| Face $i$ | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|---|
| $\mathbb{P}(i)$ | 0.05 | 0.10 | 0.15 | 0.20 | 0.25 | 0.25 |

## Task 1 · The model and its theory

1. Create numpy arrays `faces` (1 to 6) and `probs` (the table above).
2. Check that the probabilities are valid: all non-negative and summing to 1 (use `np.isclose` for the sum).
3. Compute the theoretical expectation $\mathbb{E}[X] = \sum_i i\,\mathbb{P}(i)$ as `E_theory`, the theoretical standard deviation $\sigma = \sqrt{\sum_i (i - \mathbb{E}[X])^2\,\mathbb{P}(i)}$ as `sd_theory`, and $\mathbb{P}(5)$ as `P5_theory`.

*Hint:* a weighted sum of two arrays is `faces @ probs` (Week 5).

In [ ]:
# STUDENT TASK
# Task: faces, probs, a validity check, E_theory, sd_theory, P5_theory
# Hint: faces @ probs is the sum of faces * probs
# your code here


In [ ]:
# Checkpoint 1
assert list(faces) == [1, 2, 3, 4, 5, 6], "faces should be the array 1, 2, ..., 6"
assert np.isclose(probs.sum(), 1), "probs must sum to 1"
assert np.isclose(E_theory, 4.25), "E_theory: multiply each face by its probability and add up"
assert np.isclose(sd_theory, 1.479, atol=1e-3), "sd_theory: square root of the weighted squared deviations from E_theory"
assert np.isclose(P5_theory, 0.25), "P5_theory: the probability of face 5"
print("✔ Task 1")

## Task 2 · Roll the die with the inverse-CDF method

`rng.choice(faces, p=probs)` would roll the die for you. Instead, build the die from **uniform** random numbers, as in the lecture: $Y = F_X^{-1}(U)$.

The CDF of the die is `cdf = probs.cumsum()`: `[0.05, 0.15, 0.30, 0.50, 0.75, 1.0]`. Draw $u \sim \text{Uniform}(0, 1)$. The face is **1 + the number of CDF steps that $u$ has passed**: $u = 0.12$ is past 0.05 only, so it gives face 2; $u = 0.62$ is past 0.05, 0.15, 0.30 and 0.50, so it gives face 5.

Write a function `roll_die(n, probs, rng)` that returns a numpy array of `n` rolls this way.

*Hint:* compare `u` with `cdf[:-1]` (the first five steps) so that rounding in the last value never gives a face 7. A loop over `n` values works; broadcasting is faster: `(u[:, None] > cdf[:-1])` is an `n × 5` Boolean array, and `.sum(axis=1)` counts the steps passed in each row.

In [ ]:
# STUDENT TASK
# Task: roll_die(n, probs, rng) using uniforms and the CDF
# Hint: face = 1 + number of values in cdf[:-1] that u is greater than
# your code here


In [ ]:
# Checkpoint 2
test = roll_die(100_000, probs, np.random.default_rng(0))
assert len(test) == 100_000, "roll_die(n, ...) should return n rolls"
assert set(np.unique(test)) <= {1, 2, 3, 4, 5, 6}, "every roll must be a face from 1 to 6"
assert abs((test == 6).mean() - 0.25) < 0.01, "about 25% of the rolls should be 6: check the comparison with the CDF"
assert abs((test == 1).mean() - 0.05) < 0.01, "about 5% of the rolls should be 1"
print("✔ Task 2")

In [ ]:
# If your result does not match the checkpoint, run this cell to continue.
faces = np.arange(1, 7)
probs = np.array([0.05, 0.10, 0.15, 0.20, 0.25, 0.25])
E_theory, sd_theory, P5_theory = faces @ probs, np.sqrt(((faces - faces @ probs) ** 2) @ probs), 0.25

def roll_die(n, probs, rng):
    """Roll a die with face probabilities probs n times, by the inverse-CDF method."""
    return 1 + (rng.uniform(0, 1, size=n)[:, None] > probs.cumsum()[:-1]).sum(axis=1)

## Task 3 · Histogram against PMF

1. Roll the die `N = 100_000` times and store the result in `rolls`.
2. Store the relative frequency of each face (1 to 6) in a numpy array `freq`.
3. Plot `freq` as bars and the PMF `probs` as points or a line on the same axes, with a legend and axis labels.

*Hint:* `(rolls == face).mean()` is the relative frequency of one face; a list comprehension does all six.

In [ ]:
# STUDENT TASK
# Task: rolls (N = 100_000), freq (relative frequency per face), bar chart vs PMF
# your code here


In [ ]:
# Checkpoint 3
assert len(rolls) == 100_000, "rolls should hold 100_000 rolls"
assert len(freq) == 6 and np.isclose(freq.sum(), 1), "freq: one relative frequency per face, summing to 1"
assert np.max(np.abs(freq - probs)) < 0.01, "freq should be within 0.01 of probs for every face"
print("✔ Task 3")

In [ ]:
# If your result does not match the checkpoint, run this cell to continue.
N = 100_000
rolls = roll_die(N, probs, rng)
freq = np.array([(rolls == f).mean() for f in faces])

## Task 4 · Convergence of the running mean and of $\hat P(5)$

After $k$ rolls, the running mean is $m_k = \frac1k \sum_{i \le k} x_i$, and the running estimate of $\mathbb{P}(5)$ is the running mean of the indicator "roll $i$ was a 5".

1. Compute `running_mean` and `running_p5` (both of length `N`) with `np.cumsum`.
2. Plot each against the number of rolls (two panels: `fig, axs = plt.subplots(1, 2, ...)`), with a dashed horizontal line at the theoretical value. A log-scaled x axis helps.

In [ ]:
# STUDENT TASK
# Task: running_mean and running_p5 with cumsum; two panels with the theory as dashed lines
# your code here


In [ ]:
# Checkpoint 4
assert len(running_mean) == len(rolls) and len(running_p5) == len(rolls), "one running value per roll"
assert running_mean[0] == rolls[0], "the first running mean is just the first roll"
assert abs(running_mean[-1] - E_theory) < 0.03, "after 100,000 rolls the running mean should be close to 4.25"
assert abs(running_p5[-1] - P5_theory) < 0.01, "after 100,000 rolls the estimated P(5) should be close to 0.25"
print("✔ Task 4")

In [ ]:
# If your result does not match the checkpoint, run this cell to continue.
k = np.arange(1, len(rolls) + 1)
running_mean = rolls.cumsum() / k
running_p5 = (rolls == 5).cumsum() / k

**Question A1.** Why do both curves settle on the theoretical values? And why does the first part of each curve jump around so much more than the end?

*Your answer:*

## Task 5 · A running 95 % confidence band

After $k$ rolls, the 95 % confidence interval for $\mathbb{E}[X]$ is

$$m_k \pm \Phi^{-1}(0.975)\,\frac{s_k}{\sqrt{k}}$$

where $s_k$ is the sample standard deviation of the first $k$ rolls and $\Phi^{-1}(0.975) = $ `stats.norm.ppf(0.975)` $\approx 1.96$.

1. Compute `running_sd` for $k \ge 2$ without a loop, using $s_k^2 = \dfrac{\sum_{i \le k} x_i^2 - k\,m_k^2}{k - 1}$. (Set the first value, $k = 1$, to `np.nan`.)
2. Compute `half_width` and the band `lower`, `upper`.
3. Plot the running mean with the band (`ax.fill_between(k, lower, upper, alpha=0.3)`) and the line $\mathbb{E}[X]$, from roll 10 onwards, with a log x axis.
4. Compute `coverage`: the fraction of $k$ from 100 to $N$ for which the band contains `E_theory`.

In [ ]:
# STUDENT TASK
# Task: running_sd, half_width, lower/upper band, plot, coverage
# Hint: (rolls ** 2).cumsum() gives the running sum of squares
# your code here


In [ ]:
# Checkpoint 5
assert np.isnan(running_sd[0]), "running_sd[0] (one roll) should be nan"
assert abs(running_sd[-1] - sd_theory) < 0.02, "running_sd at the end should be close to sd_theory (about 1.48)"
assert np.isclose(half_width[-1], 1.96 * running_sd[-1] / np.sqrt(len(rolls)), rtol=1e-3), \
    "half_width: 1.96 * s_k / sqrt(k): did you divide by sqrt(k)?"
assert half_width[-1] < 0.01, "after 100,000 rolls the half-width should be under 0.01"
assert 0 <= coverage <= 1, "coverage is a fraction"
print("✔ Task 5")

**Question A2.** By what factor does the band narrow between $k = 1{,}000$ and $k = 100{,}000$? The band is a 95 % interval: does that mean `E_theory` must be inside it for 95 % of the values of $k$? Look at your `coverage`.

*Your answer:*

# Part B · Real data: CO₂ per person in two regions

The lecture compared EU members with the rest of Europe in 2022. Now compare **Europe** with **South America** in **2023**, using the same Our World in Data file. The cell below keeps the countries of each region with data for 2023; `co2_per_capita` is in tonnes of CO₂ per person.

*Data: Our World in Data / Global Carbon Project, CO₂ and Greenhouse Gas Emissions dataset (CC BY 4.0).*

In [ ]:
co2 = pd.read_csv(get_data("owid_co2_w09.csv"))
EUROPE = ("ALB AND AUT BEL BGR BIH BLR CHE CZE DEU DNK ESP EST FIN "
          "FRA GBR GRC HRV HUN IRL ISL ITA LIE LTU LUX LVA MDA MKD "
          "MLT MNE NLD NOR POL PRT ROU RUS SRB SVK SVN SWE UKR").split()
SOUTH_AMERICA = "ARG BOL BRA CHL COL ECU GUY PER PRY SUR URY VEN".split()
y23 = co2[co2["year"] == 2023].dropna(subset=["co2_per_capita"])
europe = y23.loc[y23["iso_code"].isin(EUROPE), "co2_per_capita"]
samerica = y23.loc[y23["iso_code"].isin(SOUTH_AMERICA), "co2_per_capita"]
print(len(europe), "European countries,", len(samerica), "South American countries")

## Task 6 · A confidence interval for each region

1. Write a function `mean_ci(values, conf=0.95)` that returns a tuple `(mean, low, high)` using the lecture's formula $\bar x \pm \Phi^{-1}(1-\alpha/2)\,s/\sqrt{n}$, with $s$ the sample standard deviation (`ddof=1`) and $\alpha = 1 - $ `conf`.
2. Store `ci_europe = mean_ci(europe)` and `ci_samerica = mean_ci(samerica)` and print both, rounded to 2 decimals.
3. South America has only 12 countries. Store the $t$ interval for its mean as `t_samerica = stats.t.interval(0.95, df=n - 1, loc=mean, scale=se)`.

In [ ]:
# STUDENT TASK
# Task: mean_ci(values, conf=0.95) -> (mean, low, high); ci_europe, ci_samerica, t_samerica
# your code here


In [ ]:
# Checkpoint 6
assert len(ci_europe) == 3, "mean_ci should return (mean, low, high)"
assert np.allclose(ci_europe, (5.36, 4.72, 5.99), atol=0.01), "ci_europe: check ddof=1, the 1.96 and the division by sqrt(n)"
assert np.allclose(ci_samerica, (2.98, 2.23, 3.74), atol=0.01), "ci_samerica: same formula, South American data"
assert np.allclose(t_samerica, (2.13, 3.83), atol=0.01), "t_samerica: stats.t.interval with df = n - 1 and scale = standard error"
assert np.isclose(mean_ci(europe, conf=0.90)[2], 5.89, atol=0.01), "mean_ci should use conf: a 90% interval is narrower"
print("✔ Task 6")

In [ ]:
# If your result does not match the checkpoint, run this cell to continue.
def mean_ci(values, conf=0.95):
    """Mean and normal-approximation confidence interval for the mean."""
    values = np.asarray(values, dtype=float)
    se = values.std(ddof=1) / np.sqrt(len(values))
    z = stats.norm.ppf(1 - (1 - conf) / 2)
    return values.mean(), values.mean() - z * se, values.mean() + z * se

ci_europe, ci_samerica = mean_ci(europe), mean_ci(samerica)

**Question B1.** Why is the $t$ interval for South America wider than the 1.96 interval? Would switching to $t$ matter much for Europe's 41 countries? (Try `stats.t.ppf(0.975, 40)`.)

*Your answer:*

## Task 7 · Check the formula by simulation (bootstrap)

The formula relies on the sample mean being roughly normal. A **bootstrap** checks this by simulation, with no formula: treat the sample as if it were the population, draw many new samples of the same size **with replacement**, and look at how their means vary.

1. Draw `B = 5000` bootstrap samples of `europe` with `rng.choice(values, size=n, replace=True)` and store their means in an array `boot_means`.
2. Store the 2.5th and 97.5th percentiles of `boot_means` as `boot_ci` (use `np.percentile`).
3. Plot a histogram of `boot_means` with vertical lines at the formula interval `ci_europe[1]`, `ci_europe[2]`.

In [ ]:
# STUDENT TASK
# Task: boot_means (B = 5000 resampled means of europe), boot_ci (2.5 and 97.5 percentiles), histogram
# your code here


In [ ]:
# Checkpoint 7
assert len(boot_means) == 5000, "boot_means should hold one mean per bootstrap sample (B = 5000)"
assert len(boot_ci) == 2 and boot_ci[0] < boot_ci[1], "boot_ci: [2.5th percentile, 97.5th percentile]"
assert abs(boot_ci[0] - ci_europe[1]) < 0.15 and abs(boot_ci[1] - ci_europe[2]) < 0.15, \
    "the bootstrap interval should be close to the formula interval: resample with replacement, same size as the data"
print("✔ Task 7")

## Task 8 · Welch's t-test: Europe vs South America

1. State $\mathcal{H}_0$ and $\mathcal{H}_1$ in the Markdown cell below the checkpoint (as part of Question B2).
2. Store the difference of the means (Europe minus South America) as `diff`.
3. Run Welch's t-test and store the statistic and $p$-value as `t_stat`, `p_value`.
4. Store `reject` = `True` if $\mathcal{H}_0$ is rejected at $\alpha = 0.05$, else `False`.

In [ ]:
# STUDENT TASK
# Task: diff, t_stat, p_value (Welch: equal_var=False), reject at alpha = 0.05
# your code here


In [ ]:
# Checkpoint 8
assert np.isclose(diff, 2.38, atol=0.01), "diff: Europe's mean minus South America's mean"
assert np.isclose(t_stat, 4.73, atol=0.01), "t_stat: Welch's test is stats.ttest_ind(..., equal_var=False)"
assert p_value < 0.001, "p_value should be very small"
assert reject == True, "reject: compare p_value with alpha = 0.05"
print("✔ Task 8")

**Question B2.** Write down $\mathcal{H}_0$ and $\mathcal{H}_1$. Then explain in one or two sentences what your $p$-value means, in words a non-statistician would understand.

*Your answer:*

**Question B3.** A news story says: *"Statistics prove that a European emits 2.4 tonnes more CO₂ than a South American."* Give **two** reasons why this sentence over-claims, based on how the data and the test work.

*Your answer:*

## Challenge (optional) · Is the die fair? A chi-square test

A **chi-square goodness-of-fit** test compares observed counts with the counts a model expects: $\chi^2 = \sum_i (O_i - E_i)^2 / E_i$. `stats.chisquare(observed, expected)` returns the statistic and a $p$-value.

1. Roll the unfair die 600 times and count each face (`observed`).
2. Test it against **its own** probabilities (`expected = 600 * probs`): should you reject?
3. Test it against a **fair** die (`expected = 100` for each face): should you reject?
4. Repeat with only 60 rolls. What changes, and why?

In [ ]:
# STUDENT TASK
# Task: chi-square tests of 600 (then 60) rolls against the true and the fair probabilities
# your code here


## Wrap-up

In Part A you built a die from uniform random numbers with its inverse CDF, saw the law of large numbers at work, and drew a running confidence band whose width shrinks like $1/\sqrt{k}$. In Part B you used the same formula on real data, checked it with a bootstrap, compared two regions with Welch's t-test, and thought about what such a result can and cannot claim.

*Data: Our World in Data / Global Carbon Project (CC BY 4.0).*